<a href="https://colab.research.google.com/github/calvinpan1/ds3001_demandestproject/blob/main/Part2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Demand Estimation and Market Analysis: Air Fryers Part 2

In this lab, you will study the market for air fryers using brand-year data aggregated from Amazon purchases. The goal is to move from descriptive analysis to a simple demand model, and then use that model to infer markups and unit costs.

Use the cleaned file:

```python
air_fryers_clean_brand_year.csv
```

This file keeps the top 10 air-fryer brands from 2019-2023 and drops the long tail of very small brands. The variable `brand_share` has already been recomputed within this cleaned name-brand market, so shares sum to 1 within each year.

## Data

Each row is one brand in one year.

Important columns:

- `year`: calendar year
- `brand`: air-fryer brand
- `purchase_count`: number of purchases by that brand in that year
- `product_count`: number of distinct products observed for that brand-year
- `avg_price`: average price for that brand-year
- `avg_rating`: average review rating for that brand-year
- `brand_share`: purchase share within the cleaned air-fryer market in that year
- `log_brand_share`: `np.log(brand_share)`, already computed for convenience
- `compact_share`, `dual_basket_share`, `oven_style_share`, `rotisserie_share`, `window_share`: product characteristic shares for the brand-year

The original lecture wrote the demand equation using an outside option:

$$
\log(s_{bt}) - \log(s_{ot}).
$$

For this cleaned dataset, we dropped the nuisance long-tail brands instead of treating them as an outside option. You should therefore use:

$$
y_{bt} = \log(s_{bt})
$$

as the outcome and include **year dummies**. The year dummies absorb the year-specific denominator of the multinomial logit share equation. This keeps the assignment focused on the cleaned name-brand market.

## 2. Demand Estimation

We will estimate a logit-style demand model using linear regression. The model is:

$$
\log(s_{bt}) = \alpha_0 + \alpha_t + \gamma_b + \beta_{price}p_{bt} + \beta_{rating}r_{bt} + \sum_{\ell=1}^L \beta_\ell x_{bt\ell} + \epsilon_{bt}.
$$

Here:

- $b$ indexes brands
- $t$ indexes years
- $s_{bt}$ is `brand_share`
- $p_{bt}$ is `avg_price`
- $r_{bt}$ is `avg_rating`
- $x_{bt\ell}$ are the product characteristics
- $\alpha_t$ are year dummy coefficients
- $\gamma_b$ are brand dummy coefficients
- $\beta_{price}$ is **one constant price coefficient**, shared by all brands and all years

That last point matters: do **not** estimate a different price coefficient for every brand-year. We do not have enough information for that, and it would make the cost calculation impossible to interpret.

Use `pd.get_dummies(..., drop_first=True)` for brand and year dummies. The dropped brand and dropped year become the reference categories, so all dummy coefficients are interpreted relative to those omitted categories.

In [ ]:
# Load dataset
import pandas as pd

url = "https://raw.githubusercontent.com/ds4e/undergrad_ml_assignments/main/demand_est_project/air_fryers_clean_brand_year.csv"

airfryers = pd.read_csv(url)

# Save into current working directory
airfryers.to_csv("air_fryers_clean_brand_year.csv", index=False)

# Check
airfryers.head(10)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Estimate the model:

# Set y
y = airfryers["log_brand_share"]

# Get dummies
brand_dummies = pd.get_dummies(airfryers["brand"],
                               prefix="brand", drop_first=True, dtype=int)
year_dummies = pd.get_dummies(airfryers["year"].astype(str),
                              prefix="year", drop_first=True, dtype=int)
# Set feature cols
feature_cols = airfryers[["compact_share", "dual_basket_share", "rotisserie_share", "window_share"]]

X = pd.concat(
    [airfryers[["avg_price", "avg_rating"]], feature_cols,
    brand_dummies,
    year_dummies],
    axis=1,)

model = LinearRegression()
model.fit(X, y)

predicted_log_share = model.predict(X)
r2 = r2_score(y, predicted_log_share)

coef_table = pd.DataFrame({
    "feature": X.columns,
    "coefficient": model.coef_})

print("R-squared:", r2)
coef_table

R-squared: 0.7498869716880936


,feature,coefficient
0,avg_price,-0.038422
1,avg_rating,-0.917481
2,compact_share,10.651121
3,dual_basket_share,10.948041
4,rotisserie_share,-4.051675
5,window_share,8.386133
6,brand_cosori,0.919923
7,brand_cuisinart,6.365779
8,brand_dash,-0.007967
9,brand_gowise usa,2.234496



Questions:

1. What is the estimated price coefficient, $\hat{\beta}_{price}$?

$\hat{\beta}_{price}$ is -0.0384.

2. Is it negative? Why is that important?

$\hat{\beta}_{price}$ is negative, which is important because it means that there is an inverse relationship between price and product share. As price increases, product share decreases, which is exactly what the law of demand says.

3. Which product features are associated with higher demand?

Being compact, having a dual basket, and having a transparent window are associated with higher demand.

4. Which brand dummy coefficients are largest? Remember that these are interpreted relative to the dropped brand.

Cuisinart (6.36), Ninja (4.46), Instant Pot (4.12), and Oster (3.68) have the largest coefficients relative to the dropped brand of Chefman. This suggests that more luxury brand names (Cuisinart, Ninja, and Oster) and known brand names (Instant Pot, Cuisinart) tend to have a more positive impact on demand, which is exactly what we'd expect.

5. Which year dummy coefficients are largest? Remember that these are interpreted relative to the dropped year.

2020, 2021, and 2023 all have coefficients ranging from 0.11-0.12 relative to the dropped year of 2019, while 2022 had a negative coefficient of -0.04. This could be due to general consumer spending trends: 2020-2021 were strong due to COVID consumer spending increases (stimulus checks, increased consumption due to having less to do at home, more pandemic-era interest in cooking), 2022 was weaker due to a general spike in inflation and less consumer confidence, and 2023 saw this inflation spike go down.

6. What is the model's $R^2$?

The model's R^2 is 0.749, suggesting that around 3/4ths of the variation in demand is explained by the model. This model is relatively strong.